In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df, convert_col
import pickle

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /scratch/jieq/pandax/ds_notebooks/nyc-flight/src/rewritten/o4_mini_high_small/checkpoints/post_cell_15_try_3.pickle

In [ ]:
%%cudf.pandas.profile
### cell 16 ###

# Optimized for cudf

df = flights_df[flights_df["month"] == 6]
# use GPU‐native groupby + mean instead of dict‐style agg (avoids CPU fallback)
df = (
    df.groupby("carrier")[['arr_delay', 'dep_delay']]
      .mean()
      .reset_index()
)
# vectorized addition on GPU
df['total_delay'] = df['arr_delay'] + df['dep_delay']
# find the row with the smallest total_delay on GPU
best = df.nsmallest(1, 'total_delay')
# print the carrier and its delay
print(best['carrier'].iloc[0], best['total_delay'].iloc[0])

# return the dataframe
df